In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import time
import random
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ========================== CONFIGURATION ==========================
DATASET_PATH = "D:\\spring 2026\\ai lab\\project\\instacart_data"   
SAMPLE_ORDERS = 10000              # Number of orders to sample
TOP_N_PRODUCTS = 300               # Only use the most frequent N products
GRID_SIZE = 10
OBSTACLE_DENSITY = 0.3
TOP_K_ITEMS = 3
RANDOM_SEED = 42

# ========================== LOAD & FILTER DATA ==========================
print("="*60)
print("LOADING INSTACART DATASET")
print("="*60)

def load_and_filter_data(data_path, sample_orders=SAMPLE_ORDERS, top_n=TOP_N_PRODUCTS):
    #Loads Instacart CSVs, samples orders, and keeps only the top_n most frequent products.
    orders = pd.read_csv(os.path.join(data_path, "orders.csv"))
    prior_orders = orders[orders['eval_set'] == 'prior']['order_id'].unique()
    print(f"Total 'prior' orders: {len(prior_orders)}")

    if len(prior_orders) > sample_orders:
        sampled_orders = np.random.choice(prior_orders, size=sample_orders, replace=False)
    else:
        sampled_orders = prior_orders

    # Load order_products__prior (can be large, use chunks or load fully)
    op = pd.read_csv(os.path.join(data_path, "order_products__prior.csv"))
    op_sample = op[op['order_id'].isin(sampled_orders)]
    print(f"Sampled {len(sampled_orders)} orders, containing {len(op_sample)} order-product rows")

    # Find top N products by frequency in the sample
    product_counts = op_sample['product_id'].value_counts().head(top_n)
    top_products = product_counts.index.tolist()
    print(f"Keeping top {top_n} products (most frequent in sample)")

    # Filter order-product rows to only include top products
    op_filtered = op_sample[op_sample['product_id'].isin(top_products)]

    # Build basket: order_id -> list of product_ids (only top products)
    basket = op_filtered.groupby('order_id')['product_id'].apply(list).to_dict()
    # Remove orders with fewer than 2 items (cannot form pairs)
    basket = {oid: items for oid, items in basket.items() if len(items) >= 2}
    print(f"Orders with at least 2 top products: {len(basket)}")

    products = pd.read_csv(os.path.join(data_path, "products.csv"))
    products = products[products['product_id'].isin(top_products)]

    return basket, products, top_products

try:
    basket, products_df, all_products = load_and_filter_data(DATASET_PATH)
except Exception as e:
    print(f"Error: {e}")
    sys.exit(1)

# ========================== CREATE ITEM PAIR DATASET (Only Top Products) ==========================
print("\n" + "="*60)
print("CREATING ITEM ASSOCIATION DATASET")
print("="*60)

pairs = []
for items in basket.values():
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            pairs.append((items[i], items[j], 1))
            pairs.append((items[j], items[i], 1))

pair_df = pd.DataFrame(pairs, columns=['product_A', 'product_B', 'label'])
print(f"Positive pairs: {len(pair_df)}")

# Generate negative pairs among top products only
positive_set = set((a, b) for a, b, _ in pairs)
negative_pairs = []
target_neg = len(pair_df)
attempts = 0
max_attempts = target_neg * 5
while len(negative_pairs) < target_neg and attempts < max_attempts:
    a = random.choice(all_products)
    b = random.choice(all_products)
    if a != b and (a, b) not in positive_set and (b, a) not in positive_set:
        negative_pairs.append((a, b, 0))
    attempts += 1

neg_df = pd.DataFrame(negative_pairs, columns=['product_A', 'product_B', 'label'])
data = pd.concat([pair_df, neg_df], ignore_index=True)
print(f"Total dataset size: {len(data)} (balanced)")

# ========================== ONE-HOT ENCODING (Sparse) ==========================
print("\nOne-Hot Encoding with sparse matrices...")
encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
encoder.fit(np.array(all_products).reshape(-1, 1))

batch_size = 50000
n_samples = len(data)
X_A_list, X_B_list = [], []

for start in range(0, n_samples, batch_size):
    end = min(start + batch_size, n_samples)
    batch_A = encoder.transform(data['product_A'].iloc[start:end].values.reshape(-1, 1))
    batch_B = encoder.transform(data['product_B'].iloc[start:end].values.reshape(-1, 1))
    X_A_list.append(batch_A)
    X_B_list.append(batch_B)

from scipy.sparse import vstack, hstack
X_A = vstack(X_A_list)
X_B = vstack(X_B_list)
X = hstack([X_A, X_B])
y = data['label'].values

print(f"Feature matrix shape: {X.shape} (sparse)")

# ========================== TRAIN/TEST SPLIT ==========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# ========================== MODEL TRAINING ==========================
print("\n" + "="*60)
print("MACHINE LEARNING - ASSOCIATION PREDICTION")
print("="*60)

# Logistic Regression
start_time = time.time()
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr.fit(X_train, y_train)
lr_time = time.time() - start_time
y_pred_lr = lr.predict(X_test)

# Random Forest
start_time = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
rf_time = time.time() - start_time
y_pred_rf = rf.predict(X_test)

def print_metrics(name, y_true, y_pred, train_time):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name:20} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Time: {train_time:.2f}s")

print("\nModel Performance on Test Set:")
print("-"*70)
print_metrics("Logistic Regression", y_test, y_pred_lr, lr_time)
print_metrics("Random Forest", y_test, y_pred_rf, rf_time)

if f1_score(y_test, y_pred_rf) > f1_score(y_test, y_pred_lr):
    best_model = rf
    model_name = "Random Forest"
else:
    best_model = lr
    model_name = "Logistic Regression"
print(f"\nSelected model for integration: {model_name}")

GRID_SIZE = 10                     
OBSTACLE_DENSITY = 0.3            

# ========================== WAREHOUSE SIMULATION ==========================
print("\n" + "="*60)
print("WAREHOUSE PATHFINDING (A* ALGORITHM)")
print("="*60)

np.random.seed(RANDOM_SEED)
grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

num_obstacles = int(GRID_SIZE * GRID_SIZE * OBSTACLE_DENSITY)
obstacle_positions = set()
while len(obstacle_positions) < num_obstacles:
    x = np.random.randint(0, GRID_SIZE)
    y = np.random.randint(0, GRID_SIZE)
    # Keep start cell and some surrounding area free
    if (x, y) == (0, 0) or (abs(x) <= 1 and abs(y) <= 1):
        continue
    obstacle_positions.add((x, y))

for (x, y) in obstacle_positions:
    grid[x, y] = 1

def is_reachable(start, grid):
    visited = np.zeros_like(grid, dtype=bool)
    queue = [start]
    visited[start] = True
    while queue:
        cx, cy = queue.pop(0)
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = cx+dx, cy+dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and grid[nx,ny] == 0 and not visited[nx,ny]:
                visited[nx,ny] = True
                queue.append((nx,ny))
    return visited

reachable_mask = is_reachable((0,0), grid)

def find_pick_position(shelf_x, shelf_y, grid, max_radius=2):
    for r in range(1, max_radius+1):
        for dx in range(-r, r+1):
            for dy in range(-r, r+1):
                nx, ny = shelf_x + dx, shelf_y + dy
                if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and grid[nx, ny] == 0:
                    return (nx, ny)
    return None

product_location = {}
available_shelves = list(obstacle_positions)
random.shuffle(available_shelves)

for i, pid in enumerate(all_products):
    if i < len(available_shelves):
        shelf = available_shelves[i]
        pick_pos = find_pick_position(shelf[0], shelf[1], grid)
        if pick_pos is not None and reachable_mask[pick_pos]:
            product_location[pid] = pick_pos
        else:
            # Fallback: assign any reachable free cell
            free_cells = [(x,y) for x in range(GRID_SIZE) for y in range(GRID_SIZE) 
                          if grid[x,y]==0 and reachable_mask[x,y] and (x,y)!=(0,0)]
            if free_cells:
                product_location[pid] = random.choice(free_cells)
    else:
        free_cells = [(x,y) for x in range(GRID_SIZE) for y in range(GRID_SIZE) 
                      if grid[x,y]==0 and reachable_mask[x,y] and (x,y)!=(0,0)]
        if free_cells:
            product_location[pid] = random.choice(free_cells)

print(f"Warehouse grid: {GRID_SIZE}x{GRID_SIZE}, {len(obstacle_positions)} shelves")
print(f"Reachable free cells from start: {np.sum(reachable_mask)}")
print(f"Assigned pick positions for {len(product_location)} products")

print("\nGrid map (0=free, 1=obstacle, S=start):")
for i in range(GRID_SIZE):
    row = ""
    for j in range(GRID_SIZE):
        if (i,j) == (0,0):
            row += "S "
        elif grid[i,j] == 1:
            row += "# "
        elif reachable_mask[i,j]:
            row += ". "
        else:
            row += "X "
    print(row)

# ========================== A* IMPLEMENTATION ==========================
class Node:
    def __init__(self, x, y, g=float('inf'), h=0, parent=None):
        self.x = x
        self.y = y
        self.g = g
        self.h = h
        self.f = g + h
        self.parent = parent
    def __lt__(self, other):
        return self.f < other.f

def manhattan(x1, y1, x2, y2):
    return abs(x1 - x2) + abs(y1 - y2)

def get_neighbors(grid, x, y):
    for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
        nx, ny = x+dx, y+dy
        if 0 <= nx < grid.shape[0] and 0 <= ny < grid.shape[1] and grid[nx, ny] == 0:
            yield nx, ny

def a_star(grid, start, goal):
    if start == goal:
        return [start], 0
    if grid[goal] == 1:
        print(f"Warning: Goal {goal} is an obstacle!")
        return None, 0
    
    open_list = []
    closed_set = set()
    start_node = Node(start[0], start[1], g=0, h=manhattan(start[0], start[1], goal[0], goal[1]))
    heapq.heappush(open_list, start_node)
    g_values = {(start[0], start[1]): 0}
    nodes_visited = 0
    
    while open_list:
        current = heapq.heappop(open_list)
        nodes_visited += 1
        if (current.x, current.y) == goal:
            path = []
            while current:
                path.append((current.x, current.y))
                current = current.parent
            return path[::-1], nodes_visited
        if (current.x, current.y) in closed_set:
            continue
        closed_set.add((current.x, current.y))
        for nx, ny in get_neighbors(grid, current.x, current.y):
            if (nx, ny) in closed_set:
                continue
            tentative_g = current.g + 1
            if tentative_g < g_values.get((nx, ny), float('inf')):
                g_values[(nx, ny)] = tentative_g
                neighbor = Node(nx, ny, g=tentative_g, 
                                h=manhattan(nx, ny, goal[0], goal[1]), 
                                parent=current)
                heapq.heappush(open_list, neighbor)
    print(f"Warning: No path from {start} to {goal}")
    return None, nodes_visited

def multi_goal_path_planning(grid, start, goal_locations):
    current_pos = start
    remaining = set(goal_locations)
    full_path = [start]
    total_distance = 0
    total_nodes = 0
    comp_time = 0
    
    while remaining:
        nearest_goal = None
        min_dist = float('inf')
        best_path = None
        best_nodes = 0
        for goal in remaining:
            t_start = time.time()
            path, nodes = a_star(grid, current_pos, goal)
            t_elapsed = time.time() - t_start
            if path is not None:
                dist = len(path) - 1
                if dist < min_dist:
                    min_dist = dist
                    nearest_goal = goal
                    best_path = path
                    best_nodes = nodes
                    comp_time_this = t_elapsed
        if nearest_goal is None:
            print("No reachable goals remaining, stopping.")
            break
        full_path.extend(best_path[1:])
        total_distance += min_dist
        total_nodes += best_nodes
        comp_time += comp_time_this
        current_pos = nearest_goal
        remaining.remove(nearest_goal)
        print(f"  Picked goal {nearest_goal}, distance: {min_dist}")
    
    return full_path, total_distance, total_nodes, comp_time

# ========================== INTEGRATION ==========================
print("\n" + "="*60)
print("INTEGRATION: PREDICTION + PATHFINDING")
print("="*60)

def predict_associated_items(order_items, model, encoder, all_products, top_k=TOP_K_ITEMS):
    if not order_items:
        return []
    candidates = [pid for pid in all_products if pid not in order_items]
    scores = []
    a_pid = order_items[0]
    a_enc = encoder.transform([[a_pid]])
    for cand in candidates:
        b_enc = encoder.transform([[cand]])
        feat = hstack([a_enc, b_enc])
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(feat)[0][1]
        else:
            prob = model.decision_function(feat)[0]
        scores.append((cand, prob))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [cand for cand, _ in scores[:top_k]]

# Sample order (2 items)
sample_order = random.sample(all_products, 2)
print(f"Sample order items: {sample_order}")

predicted_items = predict_associated_items(sample_order, best_model, encoder, all_products, top_k=TOP_K_ITEMS)
print(f"Predicted associated items: {predicted_items}")

all_pick_items = sample_order + predicted_items
goal_coords = [product_location.get(pid) for pid in all_pick_items]
goal_coords = [g for g in goal_coords if g is not None]

print(f"Goal coordinates (free cells): {goal_coords}")

print("\nPlanning optimized route...")
opt_path, opt_dist, opt_nodes, opt_time = multi_goal_path_planning(grid, (0,0), goal_coords)
print(f"Optimized distance: {opt_dist} steps, nodes: {opt_nodes}, time: {opt_time:.4f}s")

# Baseline random order
if goal_coords:
    random_goals = list(goal_coords)
    random.shuffle(random_goals)
    print("\nPlanning baseline random order...")
    rand_path, rand_dist, rand_nodes, rand_time = multi_goal_path_planning(grid, (0,0), random_goals)
    print(f"Random order distance: {rand_dist} steps, nodes: {rand_nodes}, time: {rand_time:.4f}s")
    improvement = (rand_dist - opt_dist) / rand_dist * 100 if rand_dist > 0 else 0
else:
    rand_dist = 0
    improvement = 0
    print("No valid goals, cannot compute baseline.")

print(f"\nImprovement: {improvement:.1f}% reduction in travel distance")

# ========================== VISUALIZATION ==========================
def plot_warehouse(grid, path, goals, title="Warehouse Path"):
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.cm.colors.ListedColormap(['white', 'lightgray'])
    ax.imshow(grid.T, cmap=cmap, origin='lower')
    if path:
        path_x, path_y = zip(*path)
        ax.plot(path_x, path_y, 'b-', linewidth=2, label='Robot Path')
        ax.plot(path[0][0], path[0][1], 'go', markersize=10, label='Start')
        if len(path) > 1:
            ax.plot(path[-1][0], path[-1][1], 'ro', markersize=8, label='End')
    for g in goals:
        ax.plot(g[0], g[1], 'y*', markersize=12, markeredgecolor='black')
    ax.set_xticks(np.arange(-0.5, GRID_SIZE, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, GRID_SIZE, 1), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=0.5)
    ax.tick_params(which='minor', size=0)
    ax.set_xticks(np.arange(0, GRID_SIZE, 1))
    ax.set_yticks(np.arange(0, GRID_SIZE, 1))
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper right')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

if opt_path and len(opt_path) > 1:
    plot_warehouse(grid, opt_path, goal_coords, title=f"Optimized Route (Distance: {opt_dist})")
if rand_dist > 0:
    plot_warehouse(grid, rand_path, random_goals, title=f"Baseline Random Route (Distance: {rand_dist})")

LOADING INSTACART DATASET (Memory‑Safe)


KeyboardInterrupt: 